In [ ]:
!pip install rembg torch diffusers transformers accelerate opencv-python-headless pillow numpy

In [ ]:
import os
import io
import zipfile
import cv2
import torch
import numpy as np
from PIL import Image
from rembg import remove
from diffusers import StableDiffusionImg2ImgPipeline
from transformers import pipeline as hf_pipeline
from google.colab import files

def vivisect(image_path, layers=6):
    print("Phase 1: Stripping reality...")
    orig_pil = Image.open(image_path).convert("RGB")
    nobg_pil = remove(orig_pil)
    subject_mask = np.array(nobg_pil)[:, :, 3] > 0
    orig_cv = cv2.cvtColor(np.array(orig_pil), cv2.COLOR_RGB2BGR)

    print("Phase 2: Forging the hallucination...")
    sd_pipe = StableDiffusionImg2ImgPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16).to("cuda")
    sd_pipe.safety_checker = None
    prompt = "A high fidelity 3D render, volumetric depth, clear geometry, structural, high contrast"
    gen_pil = sd_pipe(prompt=prompt, image=orig_pil, strength=0.65, guidance_scale=7.5).images[0]
    gen_cv = cv2.cvtColor(np.array(gen_pil), cv2.COLOR_RGB2BGR)
    del sd_pipe
    torch.cuda.empty_cache()

    print("Phase 3: Bruteforce alignment...")
    gen_cv = cv2.resize(gen_cv, (orig_cv.shape[1], orig_cv.shape[0]))
    gray_src = cv2.cvtColor(orig_cv, cv2.COLOR_BGR2GRAY)
    gray_tgt = cv2.cvtColor(gen_cv, cv2.COLOR_BGR2GRAY)
    orb = cv2.ORB_create(MAX_FEATURES=5000)
    kp_src, des_src = orb.detectAndCompute(gray_src, None)
    kp_tgt, des_tgt = orb.detectAndCompute(gray_tgt, None)
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    matches = bf.knnMatch(des_src, des_tgt, k=2)
    good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]

    if len(good_matches) > 10:
        src_pts = np.float32([kp_src[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
        tgt_pts = np.float32([kp_tgt[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
        matrix, _ = cv2.findHomography(tgt_pts, src_pts, cv2.RANSAC, 5.0)
        aligned_cv = cv2.warpPerspective(gen_cv, matrix, (orig_cv.shape[1], orig_cv.shape[0])) if matrix is not None else gen_cv
    else:
        aligned_cv = gen_cv

    aligned_pil = Image.fromarray(cv2.cvtColor(aligned_cv, cv2.COLOR_BGR2RGB))

    print("Phase 4: Extracting depth map...")
    depth_pipe = hf_pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf", device=0)
    depth_array = np.array(depth_pipe(aligned_pil)["depth"]).astype(np.float32)
    del depth_pipe
    torch.cuda.empty_cache()

    print("Phase 5: Slicing strata...")
    subject_depth = depth_array[subject_mask]
    min_d, max_d = subject_depth.min(), subject_depth.max()
    normalized_depth = np.zeros_like(depth_array)
    if min_d != max_d:
        normalized_depth[subject_mask] = np.interp(depth_array[subject_mask], (min_d, max_d), (0, 255))

    bins = np.linspace(0, 255.1, layers + 1)
    zip_name = "strata.zip"
    orig_array = np.array(orig_pil)

    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        for i in range(layers):
            layer_mask = (normalized_depth >= bins[i]) & (normalized_depth < bins[i+1]) & subject_mask
            if np.any(layer_mask):
                layer_rgba = np.zeros((orig_array.shape[0], orig_array.shape[1], 4), dtype=np.uint8)
                layer_rgba[..., :3] = orig_array
                layer_rgba[..., 3] = cv2.GaussianBlur((layer_mask * 255).astype(np.uint8), (5, 5), 0)
                img_byte_arr = io.BytesIO()
                Image.fromarray(layer_rgba).save(img_byte_arr, format='PNG')
                zf.writestr(f"layer_{i:03d}.png", img_byte_arr.getvalue())

    print("Vivisection complete. Bleeding artifact to local disk...")
    files.download(zip_name)

print("Upload your victim.")
uploaded = files.upload()
for filename in uploaded.keys():
    vivisect(filename, layers=6)
